# Week 10 — Wednesday: Ridge and Lasso

**DATA 202 · Calvin University**

Monday's line learned its slopes from 2,193 houses, with no overfitting. Today it gets only **25 houses**. It still fits them, with slopes that make no sense.

> 🎯 **When a line's slopes grow too big, how do we shrink them?**

**Today's plan (~45 min):**

| Time | Part |
|---|---|
| ~6 min | 1 · Twenty-five houses |
| ~8 min | 2 · Slopes that make no sense |
| ~8 min | 3 · Ridge: a price on big slopes (SLO 10B) |
| ~6 min | 4 · Choosing alpha (SLO 10B) |
| ~9 min | 5 · Lasso: unneeded slopes go to zero (SLO 10B) |
| ~5 min | 6 · Elastic net: both at once (SLO 10B) |
| ~3 min | 7 · Zooming out: curves overfit too |

**Cues:** 🎯 Predict First · 💬 Question · 🗣️ Pair Talk · 📌 One split

<sub>Data: De Cock (2011), *Journal of Statistics Education* 19(3); from the Ames City Assessor's Office</sub>

---
## 1 · Twenty-Five Houses · ~6 min

* Suppose we only had **25 sales** to learn from, picked at random from Monday's training houses
* Monday's kind of line: one slope per column, on **18 number columns** (Monday's 19 minus `second_floor_area`, which is exactly `living_area` − `first_floor_area`)
* Scaled first, so the slopes can be compared with each other (Part 2)

In [ ]:
import pandas as pd
import plotly.express as px
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error

ames = pd.read_csv("https://cs.calvin.edu/courses/data/202/26fa/datasets/ames.csv")
X = ames.drop(columns="price")
y = ames["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=1)    # Monday's split

columns = [c for c in X.columns if c not in ("neighborhood", "year_sold", "second_floor_area")]
few = X_train.sample(25, random_state=19)[columns]          # pretend these 25 are all the sales we have
few_price = y_train[few.index]
print(len(columns), "columns:", ", ".join(columns))

In [ ]:
line = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LinearRegression())
line.fit(few, few_price)
print(f"RMSE on the 25 houses it learned from: ${root_mean_squared_error(few_price, line.predict(few)):,.0f}")

> 🎯 **Predict First:** The line misses the 25 houses by about $15,500. How much will it miss Monday's 732 test houses by?

> 📄 **Handout 1 — Twenty-five houses.** Which dots are the 25 houses the line learned from? 🗣️ Then talk with a neighbor.

In [ ]:
predicted = line.predict(X_test[columns])
print(f"RMSE on the 732 test houses: ${root_mean_squared_error(y_test, predicted):,.0f}")
print("test houses priced below $0:", (predicted < 0).sum())

fig = px.scatter(x=y_test, y=predicted, opacity=0.3, color_discrete_sequence=["gray"],
                 labels={"x": "actual price ($)", "y": "predicted price ($)"},
                 title="Predicted vs. Actual: 732 Test Houses (gray) and the 25 (blue)")
fig.add_scatter(x=few_price, y=line.predict(few), mode="markers", name="the 25 houses",
                marker=dict(color="blue", size=9))
fig.add_scatter(x=[0, 650_000], y=[0, 650_000], mode="lines", name="perfect prediction",
                line=dict(color="black", dash="dash"))
fig.show()

<details><summary>Answer</summary>

* The 25 (blue) sit on the dashed line: RMSE **$15,508**
* The test houses (gray) spray away from it: RMSE **$65,964**, four times worse. 88 of them are off by more than $100,000, and 4 get a price below $0
* That's **overfitting**: the line fits the houses it learned from, not new ones
* 🗣️ The line sets 19 numbers (18 slopes and the intercept) to fit 25 prices: almost one number per house. It has room to follow each house's quirks
</details>

---
## 2 · Slopes That Make No Sense · ~8 min

* The columns are scaled, so each slope = dollars for **one typical step up** (one standard deviation) in that column: about 500 sq ft of living area, or 1.5 quality points
* Blue: the slopes from our 25 houses. Gray: the same line fit on all 2,193 training houses

In [ ]:
everything = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LinearRegression())
everything.fit(X_train[columns], y_train)                   # the same line, on all 2,193 training houses

# line[-1] is the pipeline's last step, the LinearRegression; .coef_ holds its slopes
slopes = pd.DataFrame({"25 houses": line[-1].coef_, "all 2,193 houses": everything[-1].coef_}, index=columns)
fig = px.bar(slopes, orientation="h", barmode="group", height=650, color_discrete_sequence=["blue", "gray"],
             labels={"value": "slope ($ per typical step)", "index": "", "variable": "learned from"},
             title="One Slope per Column")
fig.update_yaxes(autorange="reversed")                      # first column at the top
fig.show()

> 📄 **Handout 2 — Slopes.** Circle slopes that look too big or point the wrong way. Cross out columns that repeat another column. 🗣️ Compare with a neighbor.

<details><summary>Answer</summary>

* **Too big:** `living_area` **+$106,337** per step, 3.5 times the 2,193-house slope. `bedrooms` −$45,883, `full_baths` −$32,014
* **The wrong way:** `quality` **−$11,605**: a better-built house sells for *less*? `basement_area` −$27,789: more basement, lower price?
* **A tug of war:** `garage_cars` **+$57,524** and `garage_area` **−$55,913**. Both measure the garage (correlation 0.91 in our 25), so they cancel on these 25 houses. On a new house the two don't match the same way, and the prediction swings
* **Repeats:** the two garage columns; `basement_area` and `first_floor_area` (the basement sits under the first floor, correlation 0.91); `rooms`, `bedrooms` and `living_area`
* This is overfitting inside a line: slopes **too big**, **fighting** each other, on columns that add **nothing new**. Reading 8's abalone did the same, at −20.8 and +20.6
</details>

> 💬 **Question:** Why would least squares choose slopes like these?

<details><summary>Answer</summary>

* Least squares has one goal: the smallest squared misses **on the 25**. It doesn't care how big the slopes get
* With near-copies like the two garage columns, many pairs of slopes fit almost equally well: +$57k and −$56k, or +$2k and +$6k. Least squares takes whichever wins on these 25 houses, even by a few dollars
* Picture the squared error as a landscape over two slopes: a long flat valley, with no clear bottom (left picture)
</details>

<img src="https://cs.calvin.edu/courses/data/202/26fa/weeks/10/images/ridge_surface.png" width="640" alt="Two 3-D surfaces of squared error over two coefficients, beta1 and beta2. Left, least squares: a long flat trough, labeled unstable. Right, ridge: a bowl with a single lowest point, labeled stable.">

---
## 3 · Ridge: a Price on Big Slopes (SLO 10B) · ~8 min

**Ridge** adds a cost for every slope's size to the least-squares error:

$$\text{minimize} \quad \sum (\text{actual} - \text{predicted})^2 \; + \; \alpha \sum b_j^2$$

* `alpha` = 0: plain least squares, Part 1's line
* A bigger `alpha` pushes every slope toward 0. A tug of war at ±$57,000 now costs a fortune
* The valley gets a bottom (right picture): one best set of slopes
* The penalty compares slopes with each other, so the columns must share one ruler: **scale first** (we did)

> 📄 **Handout 3 — Three alphas.** Label each panel: overfitting, about right, or underfitting. 🗣️ Then talk with a neighbor.

In [ ]:
slopes = pd.DataFrame(index=columns)
for alpha in [0, 10, 1000]:
    ridge = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=alpha))
    ridge.fit(few, few_price)
    slopes[alpha] = ridge[-1].coef_
    print(f"alpha = {alpha:>4} | RMSE on the 25: ${root_mean_squared_error(few_price, ridge.predict(few)):,.0f}"
          f" | on 732 test houses: ${root_mean_squared_error(y_test, ridge.predict(X_test[columns])):,.0f}")

fig = px.bar(slopes, orientation="h", facet_col="variable", height=550,
             labels={"value": "slope ($)", "index": "", "variable": "alpha"}, title="Ridge's Slopes for Three Alphas")
fig.update_yaxes(autorange="reversed")
fig.show()

<details><summary>Answer</summary>

* **alpha = 0:** Part 1's slopes, up to $106,337: **overfitting**. RMSE $15,508 on the 25, $65,964 on test houses
* **alpha = 10:** every slope under $15,000, `quality` positive again, the garage tug of war gone: **about right**. RMSE $23,191 on the 25, **$34,102** on test houses
* **alpha = 1,000:** every slope about $1,000. The line predicts between $156,000 and $215,000 for every test house, close to the 25 houses' average: **underfitting**. RMSE $75,212 and $71,678
* A bigger alpha always fits the 25 a little worse. Up to a point, it fits new houses better
* 🗣️ With 25 different houses, the alpha = 0 slopes would change the most: they follow each house. The alpha = 1,000 slopes would barely move, but they are wrong everywhere
</details>

---
## 4 · Choosing alpha (SLO 10B) · ~6 min

`alpha` is a knob, like `k` and `max_depth`. Choose it with cross-validation, on the 25 houses only.

(`ridge__alpha` means "the `alpha` of the step named `ridge`": `make_pipeline` names each step after its class.)

In [ ]:
alphas = [0.01, 0.1, 1, 10, 100, 1000]
search = GridSearchCV(make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge()),
                      {"ridge__alpha": alphas}, cv=5, scoring="neg_root_mean_squared_error", return_train_score=True)
search.fit(few, few_price)

results = pd.DataFrame({"alpha": alphas,
                        "folds it learned from": -search.cv_results_["mean_train_score"],
                        "held-out fold": -search.cv_results_["mean_test_score"]})
fig = px.line(results, x="alpha", y=["folds it learned from", "held-out fold"], markers=True, log_x=True,
              labels={"value": "RMSE ($)", "variable": "scored on"}, title="Ridge: Choosing alpha by 5-Fold CV")
fig.show()

> 📄 **Handout 4 — Choose alpha.** Circle the best alpha. Label the overfitting side and the underfitting side.

<details><summary>Answer</summary>

* **Best: alpha = 10**, held-out RMSE **$40,892**
* **Small alpha:** $5,000 to $12,000 on the folds it learned from, $60,000 to $121,000 on the held-out fold. A huge gap: **overfitting**
* **alpha = 1,000:** about $75,000 on both: **underfitting**
* This time a **larger** knob means a **simpler** model, like k in kNN. For trees, a larger `max_depth` meant a more complex one
</details>

Does CV's choice hold up on the test houses?

In [ ]:
for name, model in [("line, 25 houses", line), ("Ridge alpha = 10, 25 houses", search.best_estimator_),
                    ("line, all 2,193 houses", everything)]:
    print(f"{name:27} | RMSE on 732 test houses: ${root_mean_squared_error(y_test, model.predict(X_test[columns])):,.0f}")

<details><summary>Answer</summary>

* Ridge on 25 houses: **$34,102**, about half the plain line's $65,964
* Plain least squares on all 2,193 houses: $28,869. With 25 houses, Ridge gets most of the way there
</details>

> 📌 **One sample:** each CV fold here holds only 5 houses. Another 25 houses could pick another alpha. With so little data, the best cure is more houses.

---
## 5 · Lasso: Unneeded Slopes Go to Zero (SLO 10B) · ~9 min

**Lasso** charges for the slopes' absolute sizes instead:

$$\text{minimize} \quad \sum (\text{actual} - \text{predicted})^2 \; + \; \alpha \sum |b_j|$$

* Same idea, one difference: Lasso pushes many slopes **all the way to 0**. Those columns drop out of the model
* Same 25 houses, same 18 columns. Lasso's alphas live on a bigger scale than Ridge's

In [ ]:
rows = []
for alpha in [10, 100, 300, 1000, 3000, 10000, 30000]:
    lasso = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Lasso(alpha=alpha, max_iter=100_000))
    cv_rmse = -cross_val_score(lasso, few, few_price, cv=5, scoring="neg_root_mean_squared_error").mean()
    kept = (lasso.fit(few, few_price)[-1].coef_ != 0).sum()
    rows.append({"alpha": alpha, "columns kept (of 18)": kept, "CV RMSE ($)": round(cv_rmse)})
pd.DataFrame(rows)

<details><summary>Answer</summary>

* **Best: alpha = 10,000**, CV RMSE **$48,540**, with **8** of the 18 columns
* **alpha = 10:** all 18 columns and $207,836: almost no penalty, so Part 1's overfitting
* **alpha = 30,000:** 4 columns, $53,875: **underfitting**
* Same shape as Ridge: overfitting on the left, underfitting on the right
</details>

In [ ]:
lasso = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Lasso(alpha=10_000, max_iter=100_000))
lasso.fit(few, few_price)
slopes = pd.Series(lasso[-1].coef_, index=columns)
print("kept:   ", ", ".join(slopes[slopes != 0].index))
print("dropped:", ", ".join(slopes[slopes == 0].index))
print(f"RMSE on 732 test houses: ${root_mean_squared_error(y_test, lasso.predict(X_test[columns])):,.0f}")

<details><summary>Answer</summary>

* **Kept 8:** `living_area`, `quality`, `year_built`, `year_remodeled`, `first_floor_area`, `garage_area`, `deck_area`, `porch_area`
* **From each set of near-copies, one stays:** `garage_area` stays, `garage_cars` goes; `first_floor_area` stays, `basement_area` goes; `living_area` stays, `rooms` and `bedrooms` go
* Test RMSE **$38,523**: far better than the plain line ($65,964), a bit behind Ridge ($34,102), with less than half the columns
</details>

> 💬 **Question:** Lasso dropped `bedrooms` and both bath columns. Does that mean bathrooms don't matter to a house's price?

<details><summary>Answer</summary>

* No. A 0 means "nothing new **next to the kept columns**, in these 25 houses": bigger houses have more bathrooms, and `living_area` already says the house is big. Monday: a slope always depends on the other columns in the model
* 📌 With another 25 houses (`random_state=2`), Lasso at the same alpha keeps `bedrooms` and `full_baths`, and drops `living_area`
</details>

> "By inciting organizations to treat all data as useful, and by developing tools with a hitherto unmatched ability to find patterns, software engineers have created tools endowed with new and somewhat alien powers."
> — Fourcade & Healy, *The Ordinal Society*, ch. 3

> 💬 **Question:** Lasso kept some facts about a home and dropped others. If an assessor's office valued your home this way, what would you want to know about the columns it kept and dropped?

<details><summary>Answer</summary>

* **Why a column was dropped:** "Lasso set it to 0" isn't a reason a homeowner can contest. With another 25 sales, other columns would survive
* **Which columns could go in at all:** Monday's model also used the neighborhood. In many U.S. cities, neighborhood lines still follow old redlining maps, so a neighborhood slope can carry that history into a price
* Penalties tame wild slopes. They don't decide what *should* count in a home's value. People do
</details>

---
## 6 · Elastic Net: Both Penalties at Once (SLO 10B) · ~5 min

**Elastic net** charges both penalties, mixed:

$$\text{minimize} \quad \sum (\text{actual} - \text{predicted})^2 \; + \; \alpha \Big[ \, r \sum |b_j| \; + \; (1 - r) \sum b_j^2 \, \Big]$$

* Two knobs: `alpha` = **how strong** the penalty is; `l1_ratio` (r) = **the balance**: 0 = all Ridge, 1 = all Lasso
* Why mix? From near-copies, Lasso tends to keep one and drop the rest (Part 5: `garage_area` in, `garage_cars` out). Ridge keeps every column. Elastic net can drop columns *and* let near-copies share a slope
* **Watch the names:** in R's `glmnet`, `alpha` is the balance and `lambda` the strength. In scikit-learn, `alpha` is always the strength
* Two knobs, so `GridSearchCV` tries every pair: 7 alphas × 5 ratios = 35 models, each with 5 folds

<sub>scikit-learn's Lasso and elastic net divide the squared misses by 2 × the number of houses, and elastic net halves the Ridge part. Same idea; it only moves the scale of `alpha`.</sub>

In [ ]:
search = GridSearchCV(make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), ElasticNet(max_iter=100_000)),
                      {"elasticnet__alpha": [0.01, 0.1, 1, 10, 100, 1000, 10000],
                       "elasticnet__l1_ratio": [0.1, 0.5, 0.9, 0.99, 1]},
                      cv=5, scoring="neg_root_mean_squared_error")
search.fit(few, few_price)

cv = pd.DataFrame(search.cv_results_["params"])
cv["CV RMSE"] = -search.cv_results_["mean_test_score"] / 1000
table = cv.pivot(index="elasticnet__alpha", columns="elasticnet__l1_ratio", values="CV RMSE")
fig = px.imshow(table, x=[f"{r:g}" for r in table.columns], y=[f"{a:g}" for a in table.index], text_auto=".0f",
                color_continuous_scale="Blues_r", range_color=[30, 100], aspect="auto",
                labels={"x": "l1_ratio (0 = Ridge, 1 = Lasso)", "y": "alpha", "color": "CV RMSE ($1000s)"},
                title="Elastic Net: CV RMSE for Every Pair of Knobs")
fig.show()

best = search.best_estimator_
print(search.best_params_, f"| CV RMSE ${-search.best_score_:,.0f}")
print("columns kept:", (best[-1].coef_ != 0).sum(), "of 18")
print(f"RMSE on 732 test houses: ${root_mean_squared_error(y_test, best.predict(X_test[columns])):,.0f}")

<details><summary>Answer</summary>

* The right column (l1_ratio = 1) is Part 5's Lasso. The dark valley runs on a diagonal: the stronger the penalty, the closer to pure Lasso the best balance sits
* **Best: alpha = 100, l1_ratio = 0.99**, CV RMSE **$37,539**, the lowest today. On the test houses: **$34,928**, about the same as Ridge ($34,102)
* It keeps **all 18** columns, all with small slopes. At "99% Lasso" it still acts like Ridge. A $10,000 slope costs 10,000 in the Lasso part but 100,000,000 in the Ridge part, so 1% of Ridge outweighs 99% of Lasso. `l1_ratio` is not a plain percentage: let CV choose it
* 📌 Each fold holds 5 houses: a few thousand dollars between the best models is noise
</details>

---
## 7 · Zooming Out: Curves Overfit Too · ~3 min

Another way to give a line too many columns: powers of one column. `PolynomialFeatures(10)` turns `living_area` into area, area², …, area¹⁰, and a line on those 10 columns is a curve. Same 25 houses:

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

sizes = pd.DataFrame({"living_area": range(690, 3120, 10)})
fig = px.scatter(x=few["living_area"], y=few_price, labels={"x": "living area (sq ft)", "y": "sale price ($)"},
                 title="A Degree-10 Curve on the 25 Houses")
for name, last in [("least squares", LinearRegression()), ("Ridge, alpha = 10", Ridge(alpha=10))]:
    curve = make_pipeline(PolynomialFeatures(10, include_bias=False), StandardScaler(), last)   # area, area², ..., area¹⁰
    curve.fit(few[["living_area"]], few_price)
    fig.add_scatter(x=sizes["living_area"], y=curve.predict(sizes), name=name)
    print(f"{name:17} | biggest slope: ${abs(curve[-1].coef_).max():,.0f}")
fig.update_yaxes(range=[-300_000, 700_000])
fig.show()

<details><summary>Answer</summary>

* **Least squares:** slopes in the **trillions** that cancel each other. None of the 25 houses is between 2,296 and 3,086 sq ft, and there the curve dives to about **−$16.5 million** at 2,800 sq ft
* **Ridge:** biggest slope about $27,000, and a gentle upward bend
* Same disease, same cure: too many columns for the data → slopes too big and fighting → a penalty on big slopes. Reading 8's products of columns were another version
</details>

---
### Wrap-up

1. Few houses, many columns: least squares **overfits**. Its slopes grow **too big**, **fight** each other, and land on columns that add **nothing new**
2. **Ridge** charges α × Σ b²: every slope shrinks, none reaches 0
3. **Lasso** charges α × Σ |b|: unneeded slopes go to exactly 0, so it also chooses columns
4. **Elastic net** mixes both: `alpha` = the strength, `l1_ratio` = the balance (0 = Ridge, 1 = Lasso)
5. Choose the knobs with **cross-validation**; scale first; a weak penalty overfits, a strong one underfits 📌
6. Curves from powers of a column overfit the same way, and the same penalties fix them
7. The best cure is more data: on all 2,193 houses, plain least squares misses by $28,869